# Netflix Content Explorer

## Artificial Intelligence Minor Project

### Netflix Movies & TV Shows Analysis

This project analyzes the Netflix Titles Dataset using Python,
Pandas, and NumPy.

The application provides an interactive, menu-driven interface
that allows users to explore Netflix's catalog and answer
business-style questions about its content.

### Analysis Performed

1. Count Movies vs TV Shows
2. Number of titles added each year
3. Top 10 producing countries
4. Top 5 most common genres
5. Rating distribution
6. Oldest and newest release years
7. Search titles by keyword
8. Find content added in a specific year
9. Count total unique countries
10. Calculate release-year summary statistics using NumPy

In [ ]:
# Import Pandas for loading, cleaning, filtering and analyzing
# the Netflix dataset.
import pandas as pd

# Import NumPy for numerical and statistical calculations.
import numpy as np

# Display a message confirming that the libraries were imported.
print("Libraries imported successfully.")

In [ ]:
# Load the Netflix dataset into a Pandas DataFrame.
# The CSV file should be in the same folder as this notebook.
df = pd.read_csv("netflix_titles.csv")

print("Dataset loaded successfully!")

In [ ]:
# Display the first five rows of the dataset.
# This allows us to visually inspect the structure and contents.
df.head()

In [ ]:
# Display the last five rows of the dataset.
df.tail()

In [ ]:
# Display the number of rows and columns.
rows, columns = df.shape

print(f"Number of rows: {rows}")
print(f"Number of columns: {columns}")

# Display all column names.
print("\nColumn names:")
for column in df.columns:
    print(f"- {column}")

In [ ]:
# Display information about:
# - column names
# - data types
# - number of non-null values
# - memory usage
#
# This helps us understand what kind of cleaning is required.
df.info()


In [ ]:
# Count the number of missing values in every column.
missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

In [ ]:
# Count completely duplicated rows.
# The project requires us to check for duplicate data.
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

In [ ]:
# Generate descriptive statistics for numerical columns.
#
# In this dataset, release_year is the main numerical variable
# that will be used later for analysis.
df.describe()

In [ ]:
# ---------------------------------------------------------
# DATA CLEANING
# ---------------------------------------------------------

# Fill missing country values with "Unknown".
# This preserves the records while clearly indicating
# that the country information is unavailable.
df["country"] = df["country"].fillna("Unknown")

# Fill missing director information.
df["director"] = df["director"].fillna("Not Listed")

# Fill missing cast information.
df["cast"] = df["cast"].fillna("Not Listed")

# Fill missing rating information.
df["rating"] = df["rating"].fillna("Not Rated")

# Remove records where date_added is missing.
# The project specifically requires these rows to be dropped.
df = df.dropna(subset=["date_added"])

print("Data cleaning completed successfully.")

In [ ]:
# Remove any unnecessary spaces from the date_added values
df["date_added"] = df["date_added"].str.strip()

# Convert date_added from text/string into Pandas datetime format
df["date_added"] = pd.to_datetime(
    df["date_added"],
    format="%B %d, %Y"
)

print("date_added converted to datetime successfully.")

In [ ]:
# Extract the year from date_added
# This creates a new column called year_added

df["year_added"] = df["date_added"].dt.year

print("year_added column created successfully.")

# Display the first few rows to verify
df[["date_added", "year_added"]].head()

In [ ]:
# Check missing values again after cleaning.
print("Missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
# Display the updated dimensions of the cleaned dataset.
print(f"Rows after cleaning: {df.shape[0]}")
print(f"Columns after cleaning: {df.shape[1]}")

# Display the first five cleaned records.
df.head()

In [ ]:
def count_movies_tv():
    """
    Count the number of Movies and TV Shows in the Netflix dataset.

    This helps us understand the overall composition of
    Netflix's content catalog.
    """
    
    # Count each type using Pandas value_counts().
    counts = df["type"].value_counts()

    print("\n" + "=" * 45)
    print("MOVIES VS TV SHOWS")
    print("=" * 45)

    # Display the result as a formatted table.
    result = counts.reset_index()
    result.columns = ["Content Type", "Number of Titles"]

    print(result.to_string(index=False))


In [ ]:
def titles_added_by_year():
    """
    Count the number of Netflix titles added to the platform
    during each year.

    The year_added column was created from date_added during
    the data-cleaning stage.
    """

    # Count titles for every year and sort chronologically.
    yearly_counts = (
        df["year_added"]
        .value_counts()
        .sort_index()
    )

    print("\n" + "=" * 45)
    print("TITLES ADDED EACH YEAR")
    print("=" * 45)

    # Convert the Series into a readable table.
    result = yearly_counts.reset_index()
    result.columns = ["Year Added", "Number of Titles"]

    print(result.to_string(index=False))


In [ ]:
def top_countries():
    """
    Find the top 10 countries represented in the Netflix catalog.

    A title may be associated with multiple countries, so the
    country column is split before counting individual countries.
    """

    # Split comma-separated countries into individual values.
    countries = (
        df["country"]
        .str.split(",")
        .explode()
        .str.strip()
    )

    # Count occurrences of each country.
    country_counts = countries.value_counts().head(10)

    print("\n" + "=" * 45)
    print("TOP 10 PRODUCING COUNTRIES")
    print("=" * 45)

    result = country_counts.reset_index()
    result.columns = ["Country", "Number of Titles"]

    print(result.to_string(index=False))


In [ ]:
def top_genres():
    """
    Find the five most common genres/categories in the
    Netflix catalog.

    Since a title can belong to multiple genres, the listed_in
    column is split before counting.
    """

    # Split the comma-separated genre values.
    genres = (
        df["listed_in"]
        .str.split(",")
        .explode()
        .str.strip()
    )

    # Count each genre and keep the top five.
    genre_counts = genres.value_counts().head(5)

    print("\n" + "=" * 45)
    print("TOP 5 MOST COMMON GENRES")
    print("=" * 45)

    result = genre_counts.reset_index()
    result.columns = ["Genre", "Number of Titles"]

    print(result.to_string(index=False))


In [ ]:
def rating_distribution():
    """
    Display the distribution of Netflix content ratings.

    Examples of ratings include TV-MA, TV-14, PG-13, R and PG.
    Missing ratings were replaced with "Not Rated" during cleaning.
    """

    # Count each rating category.
    ratings = df["rating"].value_counts()

    print("\n" + "=" * 45)
    print("RATING DISTRIBUTION")
    print("=" * 45)

    result = ratings.reset_index()
    result.columns = ["Rating", "Number of Titles"]

    print(result.to_string(index=False))


In [ ]:
def release_year_range():
    """
    Find the oldest and newest release years represented
    in the Netflix dataset.
    """

    # Find the minimum and maximum release year.
    oldest_year = df["release_year"].min()
    newest_year = df["release_year"].max()

    print("\n" + "=" * 45)
    print("OLDEST AND NEWEST RELEASE YEARS")
    print("=" * 45)

    print(f"Oldest release year : {oldest_year}")
    print(f"Newest release year : {newest_year}")


In [ ]:
def search_titles():
    """
    Search Netflix titles using a keyword entered by the user.

    The search is case-insensitive so that searches such as
    'dark', 'Dark' and 'DARK' produce equivalent results.
    """

    # Ask the user for a search keyword.
    keyword = input("Enter a keyword to search for: ").strip()

    # Make sure the user entered something.
    if not keyword:
        print("Please enter a valid keyword.")
        return

    # Search the title column.
    # case=False makes the search case-insensitive.
    # na=False prevents errors if any title value is missing.
    results = df[
        df["title"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    print("\n" + "=" * 45)
    print(f"SEARCH RESULTS FOR: '{keyword}'")
    print("=" * 45)

    if results.empty:
        print("No matching titles found.")
    else:
        # Display only useful columns.
        display_columns = [
            "show_id",
            "type",
            "title",
            "release_year",
            "rating"
        ]

        print(results[display_columns].to_string(index=False))
        print(f"\nNumber of matching titles: {len(results)}")


In [ ]:
def content_added_in_year():
    """
    Display Netflix titles that were added during a year
    specified by the user.
    """

    # Ask the user for the year.
    year_input = input("Enter the year: ").strip()

    # Validate that the input contains a valid integer.
    try:
        year = int(year_input)
    except ValueError:
        print("Invalid year. Please enter a numeric year.")
        return

    # Filter the dataset using year_added.
    results = df[df["year_added"] == year]

    print("\n" + "=" * 45)
    print(f"CONTENT ADDED IN {year}")
    print("=" * 45)

    if results.empty:
        print(f"No titles were added in {year}.")
    else:
        display_columns = [
            "show_id",
            "type",
            "title",
            "date_added",
            "release_year"
        ]

        print(results[display_columns].to_string(index=False))
        print(f"\nNumber of titles added in {year}: {len(results)}")


In [ ]:
def unique_country_count():
    """
    Calculate the total number of unique countries represented
    in the Netflix catalog.

    Multiple countries in a single record are separated and
    counted individually.
    """

    # Split multiple countries into separate values.
    countries = (
        df["country"]
        .str.split(",")
        .explode()
        .str.strip()
    )

    # Count the unique country names.
    unique_countries = countries.nunique()

    print("\n" + "=" * 45)
    print("UNIQUE COUNTRY ANALYSIS")
    print("=" * 45)

    print(f"Total unique countries: {unique_countries}")


In [ ]:
def release_year_statistics():
    """
    Calculate summary statistics for release_year using NumPy.

    Statistics calculated:
    - Mean
    - Median
    - Minimum
    - Maximum
    - Standard deviation
    """

    # Convert release_year to a NumPy array.
    years = df["release_year"].dropna().to_numpy()

    # Calculate statistics using NumPy functions.
    mean_year = np.mean(years)
    median_year = np.median(years)
    minimum_year = np.min(years)
    maximum_year = np.max(years)
    standard_deviation = np.std(years)

    print("\n" + "=" * 50)
    print("RELEASE YEAR SUMMARY STATISTICS")
    print("=" * 50)

    # Use f-strings to format numerical results.
    print(f"Mean release year              : {mean_year:.2f}")
    print(f"Median release year            : {median_year:.2f}")
    print(f"Minimum release year           : {minimum_year}")
    print(f"Maximum release year           : {maximum_year}")
    print(f"Standard deviation             : {standard_deviation:.2f}")

In [ ]:
def display_menu():
    """
    Display the main menu of the Netflix Content Explorer.
    """

    print("\n")
    print("=" * 55)
    print("             NETFLIX CONTENT EXPLORER")
    print("=" * 55)

    print("1. Count of Movies vs TV Shows")
    print("2. Number of Titles Added Each Year")
    print("3. Top 10 Producing Countries")
    print("4. Top 5 Most Common Genres")
    print("5. Rating Distribution")
    print("6. Oldest and Newest Release Years")
    print("7. Search Titles by Keyword")
    print("8. Content Added in a Specific Year")
    print("9. Total Unique Countries")
    print("10. Summary Statistics on Release Year")
    print("0. Quit")

    print("=" * 55)

In [ ]:
def netflix_content_explorer():
    """
    Run the interactive Netflix Content Explorer.

    The function repeatedly displays the menu and executes
    the analysis corresponding to the user's selection.
    """

    while True:

        # Display the menu.
        display_menu()

        # Get the user's menu choice.
        choice = input("Enter your choice: ").strip()

        # -------------------------------------------------
        # OPTION 1
        # -------------------------------------------------
        if choice == "1":
            count_movies_tv()

        # -------------------------------------------------
        # OPTION 2
        # -------------------------------------------------
        elif choice == "2":
            titles_added_by_year()

        # -------------------------------------------------
        # OPTION 3
        # -------------------------------------------------
        elif choice == "3":
            top_countries()

        # -------------------------------------------------
        # OPTION 4
        # -------------------------------------------------
        elif choice == "4":
            top_genres()

        # -------------------------------------------------
        # OPTION 5
        # -------------------------------------------------
        elif choice == "5":
            rating_distribution()

        # -------------------------------------------------
        # OPTION 6
        # -------------------------------------------------
        elif choice == "6":
            release_year_range()

        # -------------------------------------------------
        # OPTION 7
        # -------------------------------------------------
        elif choice == "7":
            search_titles()

        # -------------------------------------------------
        # OPTION 8
        # -------------------------------------------------
        elif choice == "8":
            content_added_in_year()

        # -------------------------------------------------
        # OPTION 9
        # -------------------------------------------------
        elif choice == "9":
            unique_country_count()

        # -------------------------------------------------
        # OPTION 10
        # -------------------------------------------------
        elif choice == "10":
            release_year_statistics()

        # -------------------------------------------------
        # QUIT
        # -------------------------------------------------
        elif choice == "0":
            print("\nThank you for using Netflix Content Explorer!")
            print("Program terminated successfully.")
            break

        # -------------------------------------------------
        # INVALID INPUT
        # -------------------------------------------------
        else:
            print("\nInvalid choice.")
            print("Please enter a number from 0 to 10.")

In [ ]:
# Start the Netflix Content Explorer application.
netflix_content_explorer()

In [ ]:
## Sample Run — Option 1: Movies vs TV Shows
count_movies_tv()

In [ ]:
## Sample Run — Option 2: Titles Added Each Year
titles_added_by_year()

In [ ]:
## Sample Run — Option 3: Top 10 Countries
top_countries()

In [ ]:
## Sample Run — Option 4: Top 5 Genres
top_genres()

In [ ]:
## Sample Run — Option 5: Rating Distribution
rating_distribution()

In [ ]:
## Sample Run — Option 6: Oldest and Newest Release Years
release_year_range()

In [ ]:
## Sample Run — Option 7: Search by Keyword
search_titles()

In [ ]:
## Sample Run — Option 8: Content Added in a Specific Year
content_added_in_year()

In [ ]:
## Sample Run — Option 9: Unique Countries
unique_country_count()

In [ ]:
## Sample Run — Option 10: NumPy Statistics
release_year_statistics()

## Three Key Insights from the Netflix Dataset

### Insight 1: Netflix has a much larger movie collection than TV shows

The analysis shows that the Netflix catalog contains **6,131 Movies** and **2,666 TV Shows** after data cleaning. 
Therefore, Movies make up the majority of the available content. 
This indicates that the Netflix catalog represented in this dataset is more heavily focused on movies than television series.

### Insight 2: Netflix content additions increased significantly over the years

The number of titles added to Netflix increased considerably in the later years of the dataset. 
The highest number of titles was added in **2019, with 2,016 titles**, followed by **2020 with 1,879 titles** and **2021 with 1,498 titles**. 
This shows a strong expansion of the Netflix content library during this period, particularly between 2017 and 2021.

### Insight 3: Netflix has a strong international content presence

The country analysis shows that the **United States** is the largest contributor to the Netflix catalog with **3,684 titles**, 
followed by **India with 1,046 titles** and the **United Kingdom with 805 titles**. 
The dataset contains **124 unique countries**, demonstrating that Netflix's catalog includes content from a wide range of regions around the world.
The most common genre is **International Movies**, with **2,752 titles**, further highlighting the international nature of the platform's content.

## Conclusion

The Netflix Content Explorer successfully loads, cleans, and analyzes the Netflix Titles Dataset using Python, Pandas, and NumPy. 

The program provides an interactive menu through which users can examine the distribution of Movies and TV Shows, yearly content additions, major producing countries, common genres, ratings, release years, searchable titles, content added in a specific year, unique countries, and statistical summaries of release years.

The analysis shows that Movies form the majority of the Netflix catalog, content additions grew substantially in the later years of the dataset, and Netflix has a highly diverse international content library. 

The project demonstrates how data cleaning, exploratory data analysis, filtering, aggregation, string processing, and numerical statistics can be combined to extract meaningful business insights from a real-world dataset.

Overall, the Netflix Content Explorer provides a simple but effective way to understand the composition, growth, and diversity of Netflix's content catalog through an interactive Python-based analysis system.